<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">《Build a Large Language Model From Scratch》</a> 一书的补充代码，作者：<a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 从零实现字节对编码（BPE）Tokenizer -- 简化版

- 这是一个独立 Notebook，用于从零实现流行的字节对编码（BPE）分词算法；这类算法用于 GPT-2 到 GPT-4、Llama 3 等模型，这里主要用于教学目的
- 关于分词目的的更多细节，请参考[第 2 章](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb)；这里的代码是解释 BPE 算法的 bonus 材料
- OpenAI 为训练原始 GPT 模型实现的原始 BPE tokenizer 可以在[这里](https://github.com/openai/gpt-2/blob/master/src/encoder.py)找到
- BPE 算法最早在 1994 年由 Philip Gage 的论文 "[A New Algorithm for Data Compression](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf)" 中提出
- 由于计算性能更好，现在包括 Llama 3 在内的大多数项目会使用 OpenAI 开源的 [tiktoken 库](https://github.com/openai/tiktoken)；例如，它可以加载预训练的 GPT-2 和 GPT-4 tokenizer（Llama 3 模型也使用 GPT-4 tokenizer 训练）
- 上述实现和本 Notebook 中实现的区别之一是：这里还包含一个用于训练 tokenizer 的函数（用于教学目的）
- 还有一个名为 [minBPE](https://github.com/karpathy/minbpe) 的实现也支持训练，性能可能更好（这里的实现重点是教学）；与 `minbpe` 不同，我的实现还额外允许加载原始 OpenAI tokenizer 的词表和 merges

**这是一个用于教学目的的非常朴素的实现。[bpe-from-scratch.ipynb](bpe-from-scratch.ipynb) Notebook 包含一个更完善、但也更难阅读的实现，它与 tiktoken 的行为更一致。**

&nbsp;
# 1. 字节对编码（BPE）的核心思想

- BPE 的核心思想是把文本转换成整数表示（token ID），以便用于 LLM 训练（参见[第 2 章](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb)）

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/bpe-overview.webp" width="600px">

&nbsp;
## 1.1 比特和字节

- 在进入 BPE 算法之前，先介绍字节这个概念
- 可以考虑把文本转换成字节数组（毕竟 BPE 中的 B 就是 "byte"）：

In [1]:
text = "This is some text"
byte_ary = bytearray(text, "utf-8")
print(byte_ary)

bytearray(b'This is some text')


- 当我们对 `bytearray` 对象调用 `list()` 时，每个字节都会被当成一个单独元素，结果是一个由对应字节值组成的整数列表：

In [2]:
ids = list(byte_ary)
print(ids)

[84, 104, 105, 115, 32, 105, 115, 32, 115, 111, 109, 101, 32, 116, 101, 120, 116]


- 这确实是一种把文本转换成 token ID 表示的有效方式，而这种表示正是 LLM 嵌入层所需要的
- 不过，这种方法的缺点是会为每个字符创建一个 ID（短文本也会产生很多 ID！）
- 也就是说，对于一个 17 个字符的输入文本，我们必须使用 17 个 token ID 作为 LLM 的输入：

In [3]:
print("Number of characters:", len(text))
print("Number of token IDs:", len(ids))

Number of characters: 17
Number of token IDs: 17


- 如果你之前用过 LLM，可能知道 BPE tokenizer 的词表中通常有完整单词或子词的 token ID，而不是每个字符一个 token ID
- 例如，GPT-2 tokenizer 会把同一段文本（"This is some text"）分成 4 个 token，而不是 17 个：`1212, 318, 617, 2420`
- 你可以使用交互式 [tiktoken app](https://tiktokenizer.vercel.app/?model=gpt2) 或 [tiktoken 库](https://github.com/openai/tiktoken) 进行复查：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/tiktokenizer.webp" width="600px">

```python
import tiktoken

gpt2_tokenizer = tiktoken.get_encoding("gpt2")
gpt2_tokenizer.encode("This is some text")
# 打印 [1212, 318, 617, 2420]
```

- 由于一个字节由 8 个比特组成，因此单个字节可以表示 2<sup>8</sup> = 256 种可能的值，范围是 0 到 255
- 你可以执行 `bytearray(range(0, 257))` 来确认这一点；它会提示 `ValueError: byte must be in range(0, 256)`
- BPE tokenizer 通常把这 256 个值作为最开始的 256 个单字符 token；可以运行下面的代码直观检查：

```python
import tiktoken
gpt2_tokenizer = tiktoken.get_encoding("gpt2")

for i in range(300):
    decoded = gpt2_tokenizer.decode([i])
    print(f"{i}: {decoded}")
"""
打印：
0: !
1: "
2: #
...
255: �  # <---- 到这里为止是单字符 token
256:  t
257:  a
...
298: ent
299:  n
"""
```

- 注意上面第 256 和 257 项不是单字符值，而是双字符值（一个空白字符 + 一个字母）；这是原始 GPT-2 BPE tokenizer 的一个小缺点（GPT-4 tokenizer 已经改进了这一点）

&nbsp;
## 1.2 构建词表

- BPE 分词算法的目标是构建一个常见子词词表，例如 `298: ent`（例如可以出现在 *entangle, entertain, enter, entrance, entity, ...* 中），甚至完整单词，例如：

```
318: is
617: some
1212: This
2420: text
```

- BPE 算法最早在 1994 年由 Philip Gage 的论文 "[A New Algorithm for Data Compression](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf)" 中提出
- 在进入实际代码实现之前，可以把如今用于 LLM tokenizer 的形式总结如下：

&nbsp;
## 1.3 BPE 算法概要

**1. 找出高频符号对**
- 在每次迭代中扫描文本，找出最常出现的一对字节（或字符）

**2. 替换并记录**

- 用一个新的占位 ID 替换这对符号（这个 ID 尚未使用；例如如果从 0...255 开始，第一个占位 ID 可以是 256）
- 在查找表中记录这个映射
- 查找表的大小是一个超参数，也叫“词表大小”（GPT-2 的词表大小是 50,257）

**3. 重复，直到没有收益**

- 不断重复第 1 步和第 2 步，持续合并最频繁的符号对
- 当无法进一步压缩时停止（例如没有任何符号对出现超过一次）

**解压缩（解码）**

- 为了恢复原始文本，使用查找表把每个 ID 替换回对应的符号对，从而反向执行这个过程


&nbsp;
## 1.4 BPE 算法示例

### 1.4.1 编码部分的具体示例（第 1 步和第 2 步）

- 假设训练数据文本是 `the cat in the hat`，我们想基于它为 BPE tokenizer 构建词表

**迭代 1**

1. 找出高频符号对
  - 在这段文本中，"th" 出现了两次（一次在开头，一次在第二个 "e" 前面）

2. 替换并记录
  - 用一个尚未使用的新 token ID 替换 "th"，例如 256
  - 新文本变成：`<256>e cat in <256>e hat`
  - 新词表为

```
  0: ...
  ...
  256: "th"
```

**迭代 2**

1. **找出高频符号对**  
   - 在文本 `<256>e cat in <256>e hat` 中，符号对 `<256>e` 出现了两次

2. **替换并记录**  
   - 用一个尚未使用的新 token ID 替换 `<256>e`，例如 `257`。  
   - 新文本变成：
     ```
     <257> cat in <257> hat
     ```
   - 更新后的词表为：
     ```
     0: ...
     ...
     256: "th"
     257: "<256>e"
     ```

**迭代 3**

1. **找出高频符号对**  
   - 在文本 `<257> cat in <257> hat` 中，符号对 `<257> ` 出现了两次（一次在开头，一次在 “hat” 前面）。

2. **替换并记录**  
   - 用一个尚未使用的新 token ID 替换 `<257> `，例如 `258`。  
   - 新文本变成：
     ```
     <258>cat in <258>hat
     ```
   - 更新后的词表为：
     ```
     0: ...
     ...
     256: "th"
     257: "<256>e"
     258: "<257> "
     ```
     
- 以此类推

&nbsp;
### 1.4.2 解码部分的具体示例（第 3 步）

- 为了恢复原始文本，我们按这些 token ID 被引入的相反顺序，将每个 token ID 替换回对应的符号对
- 从最终压缩后的文本开始：`<258>cat in <258>hat`
- 替换 `<258>` → `<257> `：`<257> cat in <257> hat`  
- 替换 `<257>` → `<256>e`：`<256>e cat in <256>e hat`
- 替换 `<256>` → "th"：`the cat in the hat`

&nbsp;
## 2. 一个简单的 BPE 实现

- 下面用一个 Python 类实现上面描述的算法，并模拟 `tiktoken` 的 Python 用户接口
- 注意，上面的编码部分描述的是通过 `train()` 完成的原始训练步骤；不过 `encode()` 方法的工作方式类似（只是因为要处理特殊 token，所以看起来稍微复杂一些）：

1. 将输入文本拆分成单个字节
2. 反复查找并替换（合并）相邻 token（符号对）；如果它们匹配已学习到的 BPE merge，就按从高到低的优先级（也就是学习到的顺序）进行合并
3. 持续合并，直到不能再应用新的 merge
4. 最终得到的 token ID 列表就是编码输出

In [ ]:
from collections import Counter, deque
from functools import lru_cache


class BPETokenizerSimple:
    def __init__(self):
        # 将 token_id 映射到 token_str（例如 {11246: "some"}）
        self.vocab = {}
        # 将 token_str 映射到 token_id（例如 {"some": 11246}）
        self.inverse_vocab = {}
        # BPE merges 字典：{(token_id1, token_id2): merged_token_id}
        self.bpe_merges = {}

    def train(self, text, vocab_size, allowed_special={"<|endoftext|>"}):
        """
        Train the BPE tokenizer from scratch.

        Args:
            text (str): The training text.
            vocab_size (int): The desired vocabulary size.
            allowed_special (set): A set of special tokens to include.
        """

        # 预处理：把空格替换为 'Ġ'
        # 注意，Ġ 是 GPT-2 BPE 实现的一个特殊设计
        # 例如，"Hello world" 可能会被分词为 ["Hello", "Ġworld"]
        # （GPT-4 BPE 会把它分词为 ["Hello", " world"]）
        processed_text = []
        for i, char in enumerate(text):
            if char == " " and i != 0:
                processed_text.append("Ġ")
            if char != " ":
                processed_text.append(char)
        processed_text = "".join(processed_text)#

        # 用唯一字符初始化词表，如果存在 'Ġ' 也包含进去
        # 从前 256 个 ASCII 字符开始
        unique_chars = [chr(i) for i in range(256)]

        # 将 processed_text 中尚未包含的字符加入 unique_chars
        unique_chars.extend(char for char in sorted(set(processed_text)) if char not in unique_chars)

        # 可选：如果文本处理需要，确保包含 'Ġ'
        if 'Ġ' not in unique_chars:
            unique_chars.append('Ġ')

        # 现在创建 vocab 和 inverse_vocab 字典
        self.vocab = {i: char for i, char in enumerate(unique_chars)}
        self.inverse_vocab = {char: i for i, char in self.vocab.items()}#items() 方法返回一个包含字典键值对的视图对象，enumerate() 函数为每个键值对分配一个索引，从而创建了 token_id 和 token_str 之间的双向映射。

        # 添加允许的特殊 token
        if allowed_special:
            for token in allowed_special:
                if token not in self.inverse_vocab:
                    new_id = len(self.vocab)
                    self.vocab[new_id] = token
                    self.inverse_vocab[token] = new_id

        # 将 processed_text 分词为 token ID
        token_ids = [self.inverse_vocab[char] for char in processed_text]

        # BPE 第 1-3 步：反复查找并替换高频符号对
        for new_id in range(len(self.vocab), vocab_size):
            if len(token_ids) < 2:
                break
            pair_id = self.find_freq_pair(token_ids, mode="most")
            if pair_id is None:  # No more pairs to merge. Stopping training.
                break
            
            updated = self.replace_pair(token_ids, pair_id, new_id)
            if updated == token_ids:
                break

            token_ids = updated
            self.bpe_merges[pair_id] = new_id

        # 用合并后的 token 构建词表
        for (p0, p1), new_id in self.bpe_merges.items():
            merged_token = self.vocab[p0] + self.vocab[p1]
            self.vocab[new_id] = merged_token
            self.inverse_vocab[merged_token] = new_id

    def encode(self, text):
        """
        Encode the input text into a list of token IDs.

        Args:
            text (str): The text to encode.

        Returns:
            List[int]: The list of token IDs.
        """
        tokens = []
        # 将文本拆分成 token，同时保留换行符
        words = text.replace("\n", " \n ").split()  # Ensure '\n' is treated as a separate token

        for i, word in enumerate(words):
            if i > 0 and not word.startswith("\n"):
                tokens.append("Ġ" + word)  # Add 'Ġ' to words that follow a space or newline
            else:
                tokens.append(word)  # Handle first word or standalone '\n'

        token_ids = []
        for token in tokens:
            if token in self.inverse_vocab:
                # token 本身已经在词表中
                token_id = self.inverse_vocab[token]
                token_ids.append(token_id)
            else:
                # 尝试通过 BPE 处理子词分词
                sub_token_ids = self.tokenize_with_bpe(token)
                token_ids.extend(sub_token_ids)

        return token_ids

    def tokenize_with_bpe(self, token):
        """
        Tokenize a single token using BPE merges.

        Args:
            token (str): The token to tokenize.

        Returns:
            List[int]: The list of token IDs after applying BPE.
        """
        # 将 token 分成单个字符（作为初始 token ID）
        token_ids = [self.inverse_vocab.get(char, None) for char in token]
        if None in token_ids:
            missing_chars = [char for char, tid in zip(token, token_ids) if tid is None]
            raise ValueError(f"Characters not found in vocab: {missing_chars}")

        can_merge = True
        while can_merge and len(token_ids) > 1:
            can_merge = False
            new_tokens = []
            i = 0
            while i < len(token_ids) - 1:
                pair = (token_ids[i], token_ids[i + 1])
                if pair in self.bpe_merges:
                    merged_token_id = self.bpe_merges[pair]
                    new_tokens.append(merged_token_id)
                    # 为了教学目的，可以取消下面的注释：
                    # print(f"Merged pair {pair} -> {merged_token_id} ('{self.vocab[merged_token_id]}')")
                    i += 2  # Skip the next token as it's merged
                    can_merge = True
                else:
                    new_tokens.append(token_ids[i])
                    i += 1
            if i < len(token_ids):
                new_tokens.append(token_ids[i])
            token_ids = new_tokens

        return token_ids

    def decode(self, token_ids):
        """
        Decode a list of token IDs back into a string.

        Args:
            token_ids (List[int]): The list of token IDs to decode.

        Returns:
            str: The decoded string.
        """
        decoded_string = ""
        for token_id in token_ids:
            if token_id not in self.vocab:
                raise ValueError(f"Token ID {token_id} not found in vocab.")
            token = self.vocab[token_id]
            if token.startswith("Ġ"):
                # 将 'Ġ' 替换为空格
                decoded_string += " " + token[1:]
            else:
                decoded_string += token
        return decoded_string

    @lru_cache(maxsize=None)
    def get_special_token_id(self, token):
        return self.inverse_vocab.get(token, None)

    @staticmethod
    def find_freq_pair(token_ids, mode="most"):
        if len(token_ids) < 2:
            return None
        pairs = Counter(zip(token_ids, token_ids[1:]))
        if not pairs:
            return None

        if mode == "most":
            return max(pairs.items(), key=lambda x: x[1])[0]
        elif mode == "least":
            return min(pairs.items(), key=lambda x: x[1])[0]
        else:
            raise ValueError("Invalid mode. Choose 'most' or 'least'.")

    @staticmethod
    def replace_pair(token_ids, pair_id, new_id):
        dq = deque(token_ids)
        replaced = []

        while dq:
            current = dq.popleft()
            if dq and (current, dq[0]) == pair_id:
                replaced.append(new_id)
                # 移除这对符号中的第 2 个 token，第 1 个已经被替换/移除
                dq.popleft()
            else:
                replaced.append(current)

        return replaced


### 短 token 序列的边界情况处理

BPE merge 需要相邻 token 对。  
如果 token 序列少于 2 个元素，就不存在 token 对，因此 `find_freq_pair` 会返回 `None`，训练过程会正常停止。

In [21]:
tok = BPETokenizerSimple()

assert tok.find_freq_pair([]) is None
assert tok.find_freq_pair([42]) is None

tok.train("", vocab_size=270)
tok.train("H", vocab_size=270)
tok.train("He", vocab_size=270)

print("Edge-case checks passed.")

Edge-case checks passed.


- 上面的 `BPETokenizerSimple` 类包含很多代码，在这个 Notebook 里详细讨论它超出了范围；不过下一节会简要介绍用法，帮助你更好地理解这些类方法

## 3. BPE 实现 walkthrough

- 实际使用中，我强烈建议使用 [tiktoken](https://github.com/openai/tiktoken)，因为上面的实现侧重可读性和教学目的，而不是性能
- 不过它的用法和 tiktoken 大体类似，只是 tiktoken 没有训练方法
- 下面通过几个例子看看上面的 `BPETokenizerSimple` Python 代码如何工作（详细代码讨论不在本 Notebook 范围内）

### 3.1 训练、编码和解码

- 首先，把一些示例文本作为训练数据集：

In [5]:
import os
import urllib.request

if not os.path.exists("../01_main-chapter-code/the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    file_path = "../01_main-chapter-code/the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)

with open("../01_main-chapter-code/the-verdict.txt", "r", encoding="utf-8") as f: # added ../01_main-chapter-code/
    text = f.read()

- 接下来，用词表大小 1,000 初始化并训练 BPE tokenizer
- 注意，由于前面讨论过的字节值，词表大小默认已经是 255，因此这里只“学习”745 个词表条目
- 作为对比，GPT-2 词表有 50,257 个 token，GPT-4 词表有 100,256 个 token（tiktoken 中的 `cl100k_base`），GPT-4o 使用 199,997 个 token（tiktoken 中的 `o200k_base`）；相比上面的简单示例文本，它们都有大得多的训练集

In [6]:
tokenizer = BPETokenizerSimple()
tokenizer.train(text, vocab_size=1000, allowed_special={"<|endoftext|>"})

- 你可能想检查一下词表内容（但注意这会生成一个很长的列表）

In [7]:
# print(tokenizer.vocab)
print(len(tokenizer.vocab))

1000


- 这个词表是通过 742 次合并创建的（约等于 `1000 - len(range(0, 256))`）

In [8]:
print(len(tokenizer.bpe_merges))

742


- 这意味着前 256 个条目是单字符 token

- 接下来，用 `encode` 方法使用已经创建的 merges 来编码一些文本：

In [9]:
input_text = "Jack embraced beauty through art and life."
token_ids = tokenizer.encode(input_text)
print(token_ids)

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46]


In [10]:
print("Number of characters:", len(input_text))
print("Number of token IDs:", len(token_ids))

Number of characters: 42
Number of token IDs: 20


- 从上面的长度可以看到，一个 42 字符的句子被编码成了 20 个 token ID；与基于字符/字节的编码相比，输入长度大约减半了

- 注意，词表本身会在 `decode()` 方法中使用，从而把 token ID 映射回文本：

In [11]:
print(token_ids)

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46]


In [12]:
print(tokenizer.decode(token_ids))

Jack embraced beauty through art and life.


- 遍历每个 token ID，可以帮助我们更好地理解这些 token ID 如何通过词表被解码：

In [13]:
for token_id in token_ids:
    print(f"{token_id} -> {tokenizer.decode([token_id])}")

424 -> Jack
256 ->  
654 -> em
531 -> br
302 -> ac
311 -> ed
256 ->  
296 -> be
97 -> a
465 -> ut
121 -> y
595 ->  through
841 ->  ar
116 -> t
287 ->  a
466 -> nd
256 ->  
326 -> li
972 -> fe
46 -> .


- 可以看到，大多数 token ID 表示 2 字符子词；这是因为训练数据文本很短，重复词不多，而且我们使用的词表大小相对较小

- 总结一下，调用 `decode(encode())` 应该能够重现任意输入文本：

In [14]:
tokenizer.decode(tokenizer.encode("This is some text."))

'This is some text.'

&nbsp;
# 4. 结论

- 就是这样！这就是 BPE 的基本工作方式，并且包含一个用于创建新 tokenizer 的训练方法
- 希望这个简短教程对学习有所帮助；如果有问题，欢迎在[这里](https://github.com/rasbt/LLMs-from-scratch/discussions/categories/q-a)开启新的 Discussion


**这是一个用于教学目的的非常朴素的实现。[bpe-from-scratch.ipynb](bpe-from-scratch.ipynb) Notebook 包含一个更完善、但也更难阅读的实现，它与 tiktoken 的行为更一致。**